# §4.1  Reference Definitions

In [ ]:
import sys
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path('../../../../src').resolve()))
from config import get_snapshot_redshift

try:
    from utils.matplotlib_config import setconfig
    setconfig()
except ImportError:
    pass

In [ ]:
DATA_ROOT = Path('../../../../data/2pcf/scope_xi')
MODEL     = 'lc16'
SIM       = 'L800'
N_REF     = 1024   # full-box Corrfunc reference (pending — falls back to max available n)

# Quick-look config (sections 1–2)
QK_IZ        = 155
QK_MSTAR_TAG = 'mstar9.0'
QK_Z         = get_snapshot_redshift(f'iz{QK_IZ}', 'L800')

# n values to show in detail and headline plots
PLOT_N     = [2,4,8,16,32,64, 128, 256]#[2, 8, 32, 128, 256]
HEADLINE_N = [2,4,8,16,32,64, 128, 256]#[4, 16, 64, 256]
# n=128 and n=256 are infeasible for mstar_none: runtime >> cosma8 8h/16h limits
PLOT_N_MSTAR_NONE = [n for n in PLOT_N if n < 128]

print(f'Data root:   {DATA_ROOT}')
print(f'Quick-look:  iz{QK_IZ}  →  z = {QK_Z:.3f}  ({QK_MSTAR_TAG})')

## §4.1  Reference Definitions

The reference ξ(r) is the **Corrfunc full-box run** using all k sub-volume realisations simultaneously — not the SCOPE estimator at m=k. It is deterministic (seed 1000, used only for I/O ordering) and carries no sample-variance contribution from sub-volume selection.

### Simulation / model matrix

| Simulation | L (h⁻¹ Mpc) | k | Cosmology | Models | Redshifts |
|-----------|------------|---|-----------|--------|-----------|
| L800 | 542.2 | 1024 | Planck 2013 | lc16 | z≈1.5, 0.5, 0.0 (iz155, 207, 271) |
| Mill1 | 500.0 | 64 | WMAP1 | lc16, gp14 | z≈1.9, 0.0 (iz33, 63) |
| Mill2 | 100.0 | 64 | WMAP1 | lc16, gp14 | z≈1.5, 0.0 (iz40, 67) |

### Galaxy selections

| Tag | Cut | HOD character |
|-----|-----|---------------|
| `mstar_none` | all galaxies | Many satellites; large one-halo |
| `mstar9.0` | M* > 10⁹ M☉/h | Dense; strong one-halo signal |
| `mstar10.0` | M* > 10¹⁰ M☉/h | Intermediate |
| `mstar11.0` | M* > 10¹¹ M☉/h | Sparse; shot-noise-limited two-halo |

The data audit below shows what is currently available.

In [ ]:
IZ_LIST   = [155, 207, 271]
MSTAR_TAGS = ['mstar_none', 'mstar9.0', 'mstar10.0', 'mstar11.0']

print(f"{'iz':>6}  {'mstar_tag':>12}  {'n values':>40}  n_seeds(max)")
print('-' * 85)
for iz in IZ_LIST:
    for mtag in MSTAR_TAGS:
        base  = DATA_ROOT / MODEL / f'iz{iz}' / mtag
        files = sorted(base.glob(f'n*/seed*/scope_xi_{SIM}_iz{iz}.csv')) if base.is_dir() else []
        if not files:
            print(f'{iz:>6}  {mtag:>12}  (no data)')
            continue
        tmp = pd.concat([pd.read_csv(f, nrows=1) for f in files], ignore_index=True)
        ns  = sorted(tmp['n_subvol'].unique())
        max_seeds = tmp.groupby('n_subvol')['selection_seed'].nunique().max()
        has_ref   = '✓ ref' if N_REF in ns else '✗ ref missing'
        print(f'{iz:>6}  {mtag:>12}  {str(ns):>40}  {max_seeds}  {has_ref}')
